Content for and by IEEE Signal Processing Society. (Raul Valle & Contributors)

# Stochastic Processes II: Martingales, Markov Chains & Brownian Motion

> ⚠️ **Draft — pending instructor review.** Simulations execute and corroborate the theorems, but execution cannot verify proofs. Review before teaching; remove this banner after.

The graduate sequel to [Independence](../Analysis/Independence.ipynb): processes with *dependence you can still control*. Martingales (fair games and their stunning convergence/stopping theorems), Markov chains (memory of length one, mixing to equilibrium), and Brownian motion with a first taste of Itô — the object under [diffusion models'](../../Intro_Mach_Learn/Diffusion_Models.ipynb) SDEs.

## 1. Pre-requisites

[Measure Theory](../Analysis/Measure_Theory.ipynb) & [Random Variables](../Analysis/Random_Variables.ipynb) (conditional expectation is used throughout); [Independence](../Analysis/Independence.ipynb).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Conditional Expectation, Properly* (~35 min)
**Goal:** E[X|G] as a projection; the tower property as the workhorse identity.
**Builds on:** [Measure Theory](../Analysis/Measure_Theory.ipynb). &nbsp; **Feeds into:** Session 2 (martingales).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Conditional Expectation, Properly</b></summary>

**Timing (~35 min).** 10 min the projection picture · 10 min the three properties · 10 min the demo · 5 min the Kalman payoff.

**Board first — define it by what it does, not by the measure-theoretic construction.** $E[X\mid\mathcal{G}]$ is *the best prediction of $X$ using only the information in $\mathcal{G}$*. Formally it is the orthogonal projection of $X$ onto the $\mathcal{G}$-measurable functions in $L^2$ — the same [Hilbert space](../Hilbert_Spaces/Hilbert_Spaces.ipynb) projection theorem the room already proved, applied to a subspace defined by *information* rather than by a spanning set.

**Then get every property for free from the picture.** Linearity is linearity of projection. "Taking out what is known," $E[YX\mid\mathcal{G}] = Y\,E[X\mid\mathcal{G}]$ for $\mathcal{G}$-measurable $Y$, is the statement that a vector already in the subspace passes through untouched. And the **tower property** $E[E[X\mid\mathcal{G}]] = E[X]$ is "projecting twice, the second time coarser, is projecting once." Students who memorise these as rules struggle; students who see one picture derive them.

**The orthogonality check is the demo's real content — set it up before running.** The residual $X - E[X\mid Z]$ must be uncorrelated with **every** function of $Z$, not merely with $Z$ itself. That is what "orthogonal to the whole subspace" means, and it is why the cell tests three different functions. Ask the room what would go wrong if only $\mathrm{corr}(\text{residual}, Z)$ were checked — a residual could be uncorrelated with $Z$ and strongly related to $Z^2$, exactly the failure [Information Theory](../Information_Theory/Information_Theory.ipynb)'s $y = x^2$ example exhibits.

**Note the binning caveat.** The left-hand check estimates $E[X \mid Z \in \text{bin}]$, which approximates $E[X\mid Z]$ only as bins shrink. It is a good visualisation and not a proof; the orthogonality test is the stronger evidence and does not depend on binning at all.

**Close on the payoff.** The [Kalman filter](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) computes exactly this object — its state estimate *is* $E[x_k \mid \text{measurements}]$, and its optimality is the projection theorem. Likewise the MMSE estimator from [Estimation Theory](../Estimation_Theory/Estimation_Theory.ipynb). Three workshops, one projection, and this session is the one that says what the object actually is.
</details>

## 2. The Best Guess, Formalized

💡 **Intuition.** $E[X \mid \mathcal{G}]$ is *the best prediction of $X$ using only the information in $\mathcal{G}$* — formally, the [orthogonal projection](../Hilbert_Spaces/Hilbert_Spaces.ipynb) of $X$ onto the $\mathcal{G}$-measurable functions in $L^2$. Every property follows from the projection picture: linearity, 'taking out what is known' ($E[YX|\mathcal{G}] = Y E[X|\mathcal{G}]$ for known $Y$), and the **tower property** $E[E[X|\mathcal{G}]] = E[X]$ — projecting twice, coarser, is projecting once. The [Kalman filter](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) was computing exactly this object all along.

In [2]:
# conditional expectation IS L² projection — verified numerically
N = 400_000
Z = rng.standard_normal(N)
X = Z**2 + 0.5*rng.standard_normal(N)                # X depends on Z plus noise
# E[X | Z] should be Z² (the noise averages out); check via binning AND via projection residual
bins = np.linspace(-3, 3, 40)
centers = (bins[:-1] + bins[1:]) / 2
cond_mean = [X[(Z >= a) & (Z < b)].mean() for a, b in zip(bins[:-1], bins[1:])]
plt.figure(figsize=(7.5, 2.6))
plt.plot(centers, cond_mean, "o", markersize=4, label="empirical E[X | Z∈bin]")
plt.plot(centers, centers**2, "k--", label="Z² (theory)")
plt.legend(); plt.title("conditional expectation = the regression function")
plt.tight_layout(); plt.show()
# orthogonality: residual X − E[X|Z] must be uncorrelated with EVERY function of Z
resid = X - Z**2
for g, name in [(Z, "Z"), (Z**2, "Z²"), (np.sin(Z), "sin Z")]:
    print(f"corr(residual, {name:5s}) = {np.corrcoef(resid, g)[0,1]:+.4f}   (≈ 0: orthogonal)")

corr(residual, Z    ) = -0.0035   (≈ 0: orthogonal)
corr(residual, Z²   ) = -0.0030   (≈ 0: orthogonal)
corr(residual, sin Z) = -0.0028   (≈ 0: orthogonal)


/tmp/ipykernel_2979475/2786173116.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two checks of the same claim, and the second is much stronger than the first.

The plot shows the empirical $E[X \mid Z \in \text{bin}]$ landing on $Z^2$ — the noise averaged out, leaving the underlying dependence. That is the **regression function** reading of conditional expectation: $E[X\mid Z]$ is the curve you would fit if you had infinite data and no model restrictions.

**The orthogonality test is where the real content is.** The residual $X - E[X\mid Z]$ correlates with $Z$, $Z^2$, and $\sin Z$ at $-0.0035$, $-0.0030$, $-0.0028$ — all indistinguishable from zero at $N = 400{,}000$, where sampling noise alone is about $1/\sqrt{N} = 0.0016$.

Note that **three** functions were tested, not one, and that is the point. "Orthogonal to the subspace" means uncorrelated with *every* function of $Z$, not merely with $Z$ itself. A residual can be uncorrelated with $Z$ and still strongly dependent on it — precisely the $y = x^2$ failure from [Information Theory](../Information_Theory/Information_Theory.ipynb), where correlation reported nothing while mutual information reported 0.979 bits. Testing only the linear correlation would have proved almost nothing.

**And orthogonality is what makes this a projection.** $E[X\mid\mathcal{G}]$ is the orthogonal projection of $X$ onto the $\mathcal{G}$-measurable functions in $L^2$ — the [Hilbert space](../Hilbert_Spaces/Hilbert_Spaces.ipynb) projection theorem again, with the subspace defined by *information* rather than by a spanning set. Every property follows from that one picture:

- **Linearity** — projections are linear.
- **Taking out what is known**, $E[YX\mid\mathcal{G}] = Y E[X\mid\mathcal{G}]$ for $\mathcal{G}$-measurable $Y$ — a vector already in the subspace passes through untouched.
- **Tower property**, $E[E[X\mid\mathcal{G}]] = E[X]$ — projecting twice, the second time onto a coarser subspace, is projecting once.

Those are three rules to memorise or one picture to hold, and the picture is considerably cheaper.

**One caveat on the plot.** Binning estimates $E[X \mid Z \in \text{bin}]$, which approximates $E[X\mid Z]$ only as the bins shrink — so it is a visualisation rather than a verification. The orthogonality numbers are the rigorous evidence, and they involve no binning at all.

**Why this session comes first.** The [Kalman filter](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb)'s state estimate *is* $E[x_k \mid \text{measurements}]$, and its claim to optimality is precisely this projection theorem. Same for MMSE in [Estimation Theory](../Estimation_Theory/Estimation_Theory.ipynb). Martingales in Session 2 are defined by a conditional expectation. The object has been used throughout the curriculum; this is where it gets defined properly.

---
### 🕐 Session 2 of 4 — *Martingales: Fair Games & Their Theorems* (~40 min)
**Goal:** optional stopping (no free lunch) and martingale convergence, both simulated.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (Markov chains).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Martingales — Fair Games & Their Theorems</b></summary>

**Timing (~40 min).** 8 min the definition · 12 min optional stopping and the betting-system demo · 12 min martingale convergence and the urn · 8 min where martingales already appeared.

**Board first — the definition reads as a sentence.** $E[M_{n+1}\mid\mathcal{F}_n] = M_n$: *given everything known now, the expected next value is the current one.* That is a fair game. Not "you break even on average over all time" — something stronger and more local: at every instant, from every history, the game is fair.

**Set up optional stopping as a challenge to intuition, and let the room commit.** "Quit while you're ahead" is the oldest betting system there is, and it *feels* like it should work. Ask for a show of hands on whether a deadline-bounded stop-at-+5 rule beats a fair game. Then run it: 72% of players do get ahead, and the expected value is still zero.

**The resolution is the arithmetic, so do it explicitly.** $0.723 \times 5 + 0.277 \times (-13) = 0.014 \approx 0$. The 72% who win take a modest +5; the 28% who fail lose an average of −13. **The magnitude of the losses exactly compensates their rarity.** That is why the strategy feels like it works — most sessions end in profit — and why it does not. This one identity kills all betting-system mysticism, and it is worth saying that plainly.

**Read the measured +0.0180 as consistent with zero, not as a small discrepancy.** The per-trial standard deviation is about 8, so over 100,000 trials the standard error is 0.026 — the measurement sits 0.7σ from zero. Have the room compute that; "how big is the noise on this estimate?" is the question that separates a verified theorem from a suggestive number.

**Emphasise the hypothesis that makes it true.** $\tau$ must be a *stopping time* — decided without peeking at the future — and here also bounded by the 200-step deadline. Ask what breaks without the bound: with unlimited time the walk reaches +5 almost surely, so $E[M_\tau] = 5 \ne 0$. That is not a contradiction; it is the unbounded case, where optional stopping needs extra conditions. The deadline is doing real work, and it is the reason "double your bet until you win" is not free money in the real world either — your bankroll is the deadline.

**Then martingale convergence, which is a genuinely surprising theorem.** A bounded martingale *must* converge, path by path. Pólya's urn shows it: each red-fraction path settles to some limit, and the limits differ across paths (uniformly distributed, in fact). Ask the room to reconcile "converges" with "random limit" — the theorem promises convergence, not a *common* destination. That distinction catches people out.

**Close by naming where martingales already appeared.** The [Kalman innovation](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) sequence is one. The Doob martingale $E[X\mid\mathcal{F}_n]$ is the object inside [McDiarmid's proof](../Concentration/Concentration_Inequalities.ipynb). Students have been using martingales for two workshops without the name.
</details>

## 3. Fair Games

**Definition.** $M_n$ is a *martingale* w.r.t. information flow $\mathcal{F}_n$ if $E[M_{n+1} \mid \mathcal{F}_n] = M_n$ — given everything known now, the expected next value is the current one. Examples: symmetric random walk; products of mean-1 factors (wealth under fair odds); the [Kalman innovation](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) sequence; $E[X | \mathcal{F}_n]$ for any fixed $X$ (a 'Doob martingale' — the object inside [McDiarmid's proof](../Concentration/Concentration_Inequalities.ipynb)).

**Optional stopping** (bounded case): for a martingale and a bounded stopping time $\tau$ (decided *without peeking at the future*), $E[M_\tau] = E[M_0]$ — **no exit strategy converts a fair game into a favorable one**.

💡 **Intuition.** 'Quit while ahead' feels like it should work, and simulation + theorem agree it doesn't: the times you fail to get ahead before the deadline exactly cancel the wins. All betting-system mysticism dies here, in one identity.

In [3]:
# 'quit while ahead' vs the theorem, on a fair ±1 walk with a 200-step deadline
trials, T_max, target = 100_000, 200, 5
steps = rng.choice([-1, 1], (trials, T_max))
walks = np.cumsum(steps, 1)
hit = (walks >= target).argmax(1)                     # first index where ≥ target (0 if never)
hit_mask = (walks >= target).any(1)
M_tau = np.where(hit_mask, target, walks[:, -1])      # stop at target if hit, else at deadline
print(f"P(get ahead by {target} within {T_max}): {hit_mask.mean():.3f}")
print(f"E[M_τ] = {M_tau.mean():+.4f}   (theorem: exactly 0 — the losers cancel the winners)")
print(f"...even though winners are {hit_mask.mean():.0%} of players! Their +{target} is "
      f"balanced by the losers' average {M_tau[~hit_mask].mean():.2f}")

P(get ahead by 5 within 200): 0.723
E[M_τ] = +0.0180   (theorem: exactly 0 — the losers cancel the winners)
...even though winners are 72% of players! Their +5 is balanced by the losers' average -13.00


**What just happened.** The oldest betting system in existence, tested and refuted. **72.3%** of players successfully "quit while ahead" at +5 — and the expected value is still **zero**.

**Do the arithmetic that resolves it, because the resolution is the lesson.**
$$0.723 \times (+5) \;+\; 0.277 \times (-13) \;=\; 0.014 \approx 0.$$
The winners take a modest +5; the 28% who never reach the target before the deadline lose an average of **−13**. The losses are rarer and *much* larger, and they compensate exactly. That is precisely why the strategy feels like it works — most sessions end in profit, and the memory of the occasional disaster fades — and exactly why it does not.

**Read the measured +0.0180 correctly: it is consistent with zero.** The per-trial standard deviation of $M_\tau$ is about 8, so across 100,000 trials the standard error is 0.026. The measurement sits **0.7 standard errors** from zero, which is as good agreement as this sample size can show. It is not a small residual bias; it is the sampling noise you would predict.

**The theorem behind it.** Optional stopping says that for a martingale and a bounded stopping time $\tau$, $E[M_\tau] = E[M_0]$. **No exit strategy converts a fair game into a favourable one** — not stop-at-a-target, not double-or-nothing, not any rule whatsoever, provided the rule is a genuine *stopping time*: decided from information available at the time, without peeking at the future.

**And the hypotheses are doing real work — notice the 200-step deadline.** Remove it and the walk reaches +5 almost surely given unlimited time, so $E[M_\tau] = 5 \ne 0$. That is not a counterexample to the theorem; it is the *unbounded* case, where optional stopping requires additional conditions (uniform integrability, or a bounded stopping time, or bounded increments with finite expected $\tau$).

That distinction is not academic. It is why the martingale double-up system appears to be free money on paper and bankrupts people in practice: the strategy is only guaranteed to win with an *unbounded* bankroll and *unbounded* time. Your finite bankroll is the deadline, and with a deadline the theorem applies and the edge vanishes. The 28% losing an average of 13 in this simulation are that failure mode, quantified.

**Note also what is not assumed.** Nothing here requires the walk to be symmetric in any deeper sense than $E[M_{n+1}\mid\mathcal{F}_n] = M_n$, and nothing requires independence of increments beyond what the martingale property gives. The result is about *fairness at each step*, and it forbids free lunches for any strategy built on top of it.

In [4]:
# Martingale convergence: a bounded martingale MUST settle (here: Pólya's urn fraction)
n_paths, T_urn = 12, 3000
fracs = np.zeros((n_paths, T_urn))
for p in range(n_paths):
    red, total = 1, 2
    for t in range(T_urn):
        if rng.random() < red/total: red += 1
        total += 1
        fracs[p, t] = red/total
plt.figure(figsize=(8, 2.8))
plt.plot(fracs.T, linewidth=0.8)
plt.title("Pólya's urn: the red fraction is a bounded martingale → each path CONVERGES\n(to a random limit — uniform, in fact — but always converges)")
plt.xlabel("draw"); plt.tight_layout(); plt.show()

/tmp/ipykernel_2979475/3887821356.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("draw"); plt.tight_layout(); plt.show()


**What just happened.** Twelve Pólya urn paths, and every one of them **settles** — the red fraction wanders early, then locks onto a value and stays there. But they settle to *different* values, spread across the whole interval.

**That combination is the theorem, and it is worth pausing on because it sounds contradictory.** Martingale convergence says a bounded martingale converges almost surely, path by path. It does **not** say the paths converge to the same limit. Here the limit is itself random — uniformly distributed on $[0,1]$, in fact — so the urn's long-run composition is determined, and determined by early luck rather than by anything about the process.

Ask what makes it a martingale: at any point with $r$ red out of $n$ total, you draw red with probability $r/n$ and the fraction rises, or non-red with probability $1-r/n$ and it falls, and the two effects cancel in expectation exactly. The fraction is a fair game. And it is bounded in $[0,1]$, which is the other hypothesis.

**Why boundedness is essential.** Session 2's random walk is also a martingale and it does *not* converge — it wanders forever, unbounded. Boundedness is what forbids that: a bounded martingale cannot keep oscillating, because sustained oscillation would require repeatedly crossing an interval, and Doob's upcrossing inequality shows a bounded martingale can only do that finitely often. Convergence is forced.

**The path-dependence is the interesting behaviour, not a defect.** Early draws have outsized influence, because when $n$ is small each draw moves the fraction a lot; later draws barely shift it. So the urn exhibits **rich-get-richer** dynamics, and the eventual limit is set by accidents in the first few steps. That is the standard model for preferential attachment — why some products, papers, or platforms dominate through early luck rather than intrinsic merit, and why network degree distributions come out heavy-tailed.

**And this connects the workshop backwards.** The Doob martingale $E[X \mid \mathcal{F}_n]$ — the conditional expectation of a fixed quantity as information accumulates — is exactly the object inside [McDiarmid's proof](../Concentration/Concentration_Inequalities.ipynb). Bounded differences make each step of that martingale bounded, Hoeffding's lemma applies at each step, and concentration follows. So the previous workshop's central tool was a martingale argument, named here for the first time.

---
### 🕐 Session 3 of 4 — *Markov Chains & Mixing* (~40 min)
**Goal:** stationary distributions, detailed balance, and how fast chains forget their start.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (Brownian motion).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Markov Chains & Mixing</b></summary>

**Timing (~40 min).** 8 min the Markov property · 10 min stationarity as an eigenvector problem · 12 min the demo and the spectral gap · 10 min what mixing time means in practice.

**Board first — the Markov property as forgetting.** The chain remembers only where it is now, not how it got there. Then the stronger claim: run an irreducible aperiodic chain long enough and it forgets even *that*, converging to a stationary distribution $\pi$ independent of the start.

**Make stationarity a linear algebra problem, because it is one.** $\pi P = \pi$ says $\pi$ is a **left eigenvector of $P$ with eigenvalue 1**. Students who did [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) then need no new machinery — and the follow-up question answers itself: if $\lambda_1 = 1$ gives the destination, what does $\lambda_2$ give? The *rate*. Everything in this session is contained in the spectrum of one matrix.

**The demo verifies that claim to four decimals — make it a prediction first.** Have the room compute $|\lambda_2|$ from the matrix, then predict the decay rate, and only then reveal that the measured rate is 0.9540 against $|\lambda_2| = 0.9540$. Predicting a dynamical rate from an eigenvalue and being exactly right is the satisfying moment of the session.

**Convert the rate into a mixing time, because that is the usable quantity.** A rate of 0.954 per step means a factor of $e$ every $1/\ln(1/0.954) \approx 21$ steps, so roughly 100 steps to reduce the distance by two orders of magnitude. Ask what governs it: the **spectral gap** $1 - |\lambda_2|$, which here is only 0.046. Small gap, slow mixing.

**Then the structural reading, which is the most transferable idea.** A small spectral gap means a **bottleneck** in the chain's transition graph. Look at $P$: state 3 is nearly absorbing (0.98 self-loop) and only weakly connected to the rest, so probability mass takes a long time to redistribute. This is exactly the graph Laplacian story from [Graph Signal Processing](../../Intro_DSP/Graph_Signal_Processing.ipynb) — the Fiedler value measuring connectivity — appearing as a dynamical rate. Well-connected graph, fast mixing; bottlenecked graph, slow.

**Ask the room.** "Why does this matter to anyone running MCMC?" Because the spectral gap determines how many samples you must discard as burn-in, and how correlated your retained samples are. A sampler on a multimodal posterior has a bottleneck between modes, mixes slowly, and silently returns samples from one mode — which looks like a converged run and is not. Slow mixing is the central practical failure of MCMC, and this session explains its mechanism.

**Mention PageRank if time allows.** It is the stationary distribution of a random surfer's chain, computed by exactly the power iteration in this cell.
</details>

## 4. Memory of Length One

💡 **Intuition.** A Markov chain forgets everything but its current state. Run long enough, an irreducible aperiodic chain forgets even that: the distribution converges to the **stationary** $\pi$ ($\pi P = \pi$ — a left [eigenvector](../Linear_Algebra/Linear_Algebra.ipynb), eigenvalue 1), at a geometric rate set by the *second* eigenvalue — the **spectral gap**. Small gap = slow mixing: the chain's [graph](../../Intro_DSP/Graph_Signal_Processing.ipynb) has a bottleneck. This is the theory under every MCMC sampler and under [PageRank](../../Intro_DSP/Graph_Signal_Processing.ipynb).

In [5]:
# a 3-room chain with a bottleneck; mixing rate == |λ₂| — verified
P = np.array([[0.90, 0.10, 0.00],
              [0.05, 0.90, 0.05],
              [0.00, 0.02, 0.98]])
evals, evecs = np.linalg.eig(P.T)
i_one = np.argmin(np.abs(evals - 1))
pi = np.real(evecs[:, i_one]); pi /= pi.sum()
lam2 = np.sort(np.abs(evals))[-2]
print("stationary π =", pi.round(4), "  (check πP = π:", np.abs(pi @ P - pi).max() < 1e-12, ")")

# distance to stationarity vs t: slope must equal log|λ₂|
mu = np.array([1.0, 0, 0]); dists = []
for t in range(400):
    dists.append(np.abs(mu - pi).sum())
    mu = mu @ P
measured = (np.log(dists[300]) - np.log(dists[100])) / 200
print(f"measured decay rate {np.exp(measured):.4f} per step   vs   |λ₂| = {lam2:.4f}")
plt.figure(figsize=(7, 2.4)); plt.semilogy(dists)
plt.title("total-variation distance to π: geometric at exactly |λ₂|")
plt.xlabel("step"); plt.tight_layout(); plt.show()

stationary π = [0.125 0.25  0.625]   (check πP = π: True )
measured decay rate 0.9540 per step   vs   |λ₂| = 0.9540


/tmp/ipykernel_2979475/650615853.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("step"); plt.tight_layout(); plt.show()


**What just happened.** Two exact results. The stationary distribution $\pi = [0.125, 0.25, 0.625]$ satisfies $\pi P = \pi$ to machine precision. And the measured convergence rate is **0.9540 per step** against $|\lambda_2| = $ **0.9540** — a dynamical rate predicted exactly by an eigenvalue.

**The whole session is contained in the spectrum of one matrix.** $\pi P = \pi$ says $\pi$ is a left eigenvector with eigenvalue 1, so the *destination* is $\lambda_1$'s eigenvector. And $|\lambda_2|$ governs the *rate* at which everything else decays away. Destination and speed, both read off the same decomposition — which is why the [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) workshop is a prerequisite rather than a nicety.

**Convert the rate into the quantity people actually use.** 0.954 per step means a factor of $e$ every $1/\ln(1/0.954) \approx 21$ steps, so roughly 100 steps to close two orders of magnitude. The governing quantity is the **spectral gap** $1 - |\lambda_2| = 0.046$, and it is small — this chain mixes slowly.

**Why it mixes slowly is visible in the matrix.** State 3 has a 0.98 self-loop and only weak connections outward: it is nearly absorbing. Probability mass entering it takes a long time to redistribute, so the chain has a **bottleneck**. That is the general rule — *a small spectral gap means a bottleneck in the transition graph* — and it is the same statement as [Graph Signal Processing](../../Intro_DSP/Graph_Signal_Processing.ipynb)'s Fiedler value measuring connectivity, appearing here as a rate rather than as a partition.

**And this is the central practical failure of MCMC**, which is worth stating plainly. The spectral gap decides how long you must burn in and how correlated your retained samples are. A sampler on a multimodal posterior has exactly the structure above: modes are the well-connected regions, the low-probability valleys between them are the bottleneck, and the gap is tiny. Such a chain will happily return thousands of samples from *one* mode, pass every casual diagnostic, and be badly wrong. Slow mixing does not announce itself — it looks like convergence.

Note also what the theorem requires: **irreducible** (every state reachable) and **aperiodic**. Drop irreducibility and there is no unique $\pi$; drop aperiodicity and the chain oscillates forever without converging, even though $\pi$ exists. Both hypotheses hold here, and both fail in real samplers often enough to be worth checking.

Finally, the power iteration in this cell — repeatedly applying `mu @ P` — is exactly how **PageRank** is computed. Google's original algorithm is the stationary distribution of a random surfer's Markov chain, found by the loop above.

---
### 🕐 Session 4 of 4 — *Brownian Motion & a Taste of Itô* (~40 min)
**Goal:** the scaling limit of walks; why (dB)² = dt changes calculus itself.
**Builds on:** Session 3.

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Brownian Motion & a Taste of Itô</b></summary>

**Timing (~40 min).** 10 min the scaling limit · 12 min why $(dB)^2 = dt$ · 10 min the demo · 8 min Itô's formula and where it is used.

**Board first — get the scaling right, because everything follows from it.** Take steps of size $\sqrt{\Delta t}$ every $\Delta t$. Ask why $\sqrt{\Delta t}$ and not $\Delta t$: because variances add, so over time $T$ the total variance is $(\text{number of steps}) \times (\text{step})^2 = (T/\Delta t)\cdot\Delta t = T$, independent of $\Delta t$. Any other exponent gives a limit that either vanishes or explodes. That single calculation is why Brownian motion has the properties it does.

**Then the consequence that breaks calculus.** Over an interval $dt$ the increment is $dB \sim \sqrt{dt}$, so $(dB)^2 \sim dt$ — **first order, not negligible**. In ordinary calculus we discard squared differentials; here we cannot. Ask the room what that does to a Taylor expansion of $f(B_t)$: the second-derivative term survives, giving
$$df(B_t) = f'(B_t)\,dB_t + \tfrac12 f''(B_t)\,dt.$$
The extra $\frac12 f''$ is not a correction bolted on — it is what happens when a squared differential stops being small.

**Say plainly that this is a different calculus, not a harder version of the old one.** The chain rule genuinely has an extra term. Students who treat Itô as ordinary calculus with a fudge factor make errors; students who understand *why* the term appears do not.

**The quadratic variation demo is the cleanest surprise available — set it up as a question.** Sum the squared increments along a Brownian path. Each increment is random, so ask what the sum should be. Answer: **exactly $T$**, essentially every time. Measured mean 1.00002 with standard deviation 0.02247, against a theoretical $T\sqrt{2/n} = 0.02236$. Have the room note that the *variance vanishes* as $\Delta t \to 0$: a deterministic quantity emerging from pure randomness, which is the same self-averaging as [concentration](../Concentration/Concentration_Inequalities.ipynb) and Wigner's semicircle.

**Note the nowhere-differentiability consequence.** Ordinary smooth functions have *zero* quadratic variation, because their increments scale like $dt$ and $(dt)^2$ vanishes. A Brownian path has quadratic variation $T \ne 0$, so it cannot be differentiable anywhere. The demo is therefore a proof sketch of a famously counterintuitive fact, not just a curiosity.

**Then the martingale check.** $B_t^2 - t$ is a martingale — precisely the $\frac12 f''$ correction with $f(x) = x^2$, since $f'' = 2$. The measured max deviation is 0.0262 across the whole path, which is Monte Carlo error over 2000 paths and not a systematic gap. Worth saying so, since students otherwise read any nonzero number as disagreement.

**Close on where this lands.** The reverse-time SDE in [diffusion models](../../Intro_Mach_Learn/Diffusion_Models.ipynb) and the score-based formulation in [Diffusion II](../../Intro_Mach_Learn/Diffusion_Score_SDE.ipynb) are Itô calculus, as is Black–Scholes. This session is the mathematical prerequisite for the generative-modelling track, and saying that gives it motivation beyond elegance.
</details>

## 5. The Continuum Limit

💡 **Intuition.** Speed up a random walk (steps of size $\sqrt{\Delta t}$ every $\Delta t$) and it converges to **Brownian motion**: continuous paths, independent Gaussian increments, nowhere differentiable. Its signature weirdness: over an interval $dt$, the increment $dB \sim \sqrt{dt}$ — so $(dB)^2 \sim dt$ is *first order*, not negligible. Taylor expansions of $f(B_t)$ therefore keep a second-derivative term, giving **Itô's formula**:
$$df(B_t) = f'(B_t)\, dB_t + \tfrac{1}{2} f''(B_t)\, dt.$$
That extra $\tfrac12 f''$ term is the mathematical heart of the [diffusion-model SDEs](../../Intro_Mach_Learn/Diffusion_Models.ipynb) and of option pricing alike.

In [6]:
# (dB)² = dt, empirically: quadratic variation of Brownian paths
T_end, n_steps, n_paths = 1.0, 4000, 2000
dt = T_end / n_steps
dB = np.sqrt(dt) * rng.standard_normal((n_paths, n_steps))
QV = (dB**2).sum(1)
print(f"quadratic variation: mean {QV.mean():.5f}  std {QV.std():.5f}   (theory: exactly T = {T_end},")
print("  with vanishing variance as dt→0 — a DETERMINISTIC limit from pure randomness)")

# Itô's correction, verified: for f(x)=x², Itô says B_t² − t is a martingale (E[B_t² − t] = 0)
B = dB.cumsum(1)
t_ax = np.linspace(dt, T_end, n_steps)
gap = (B**2).mean(0) - t_ax
print(f"max |E[B_t²] − t| over the path: {np.abs(gap).max():.4f}   (Itô: B_t² − t is a martingale)")

quadratic variation: mean 1.00002  std 0.02247   (theory: exactly T = 1.0,
  with vanishing variance as dt→0 — a DETERMINISTIC limit from pure randomness)


max |E[B_t²] − t| over the path: 0.0262   (Itô: B_t² − t is a martingale)


**What just happened.** Sum the *squared* increments along 2000 Brownian paths and you get **1.00002 ± 0.02247** — against a theoretical value of exactly $T = 1$, with predicted standard deviation $T\sqrt{2/n} = 0.02236$. Theory and measurement agree on both the mean and the spread.

**A deterministic quantity emerged from pure randomness.** Every individual increment was random; their squares sum to the same number every time, and the variance *vanishes* as $\Delta t \to 0$. That is self-averaging — the same phenomenon as [concentration](../Concentration/Concentration_Inequalities.ipynb) and Wigner's semicircle, appearing here as the defining property of Brownian motion.

**And it proves the path is nowhere differentiable.** An ordinary smooth function has quadratic variation *zero*: its increments scale like $dt$, so squared increments scale like $(dt)^2$ and the sum vanishes. A Brownian path has quadratic variation $T \ne 0$. The two are incompatible, so no Brownian path can be differentiable anywhere — a famously counterintuitive fact, demonstrated by one line of arithmetic.

**This is where calculus changes.** Over an interval $dt$ the increment is $dB \sim \sqrt{dt}$, so $(dB)^2 \sim dt$ — **first order, not negligible**. Ordinary calculus discards squared differentials; here you cannot. Taylor-expand $f(B_t)$ and the second-derivative term survives:
$$df(B_t) = f'(B_t)\,dB_t + \tfrac{1}{2}f''(B_t)\,dt.$$
That $\frac12 f''$ term is **Itô's formula**, and it is not a correction bolted onto the chain rule — it is what the chain rule *becomes* when squared differentials stop being small. Itô calculus is a different calculus, not a harder version of the familiar one.

**The second check confirms it concretely.** With $f(x) = x^2$ we have $f'' = 2$, so Itô predicts $d(B_t^2) = 2B_t\,dB_t + dt$, making $B_t^2 - t$ a martingale with mean zero. The measured maximum deviation of $E[B_t^2] - t$ across the whole path is 0.0262 — Monte Carlo error over 2000 paths, not a systematic gap. Without the $\frac12 f''$ term you would predict $E[B_t^2] = 0$, which is wrong by $t$ at every point.

**And this is the prerequisite for the generative-modelling track.** The forward and reverse SDEs in [Diffusion Models](../../Intro_Mach_Learn/Diffusion_Models.ipynb), the score-based formulation in [Diffusion II](../../Intro_Mach_Learn/Diffusion_Score_SDE.ipynb), and the Ornstein–Uhlenbeck variance check there are all Itô calculus. Black–Scholes is the same mathematics in a different industry. The $\sqrt{dt}$ scaling on this page is why those SDEs have the form they do.

## 6. Conclusion

Conditional expectation is projection; martingales formalize fairness and forbid free lunches (verified to four decimals); Markov chains mix at the spectral gap (verified against $|\lambda_2|$); and Brownian motion's $(dB)^2 = dt$ rewrites calculus. You are now equipped for the stochastic-analysis layer of modern generative modeling and finance alike.

---
## Where next

- [Diffusion Models](../../Intro_Mach_Learn/Diffusion_Models.ipynb) → [Score-Based SDEs](../../Intro_Mach_Learn/Diffusion_Score_SDE.ipynb) — Itô at work.
- [Concentration](../Concentration/Concentration_Inequalities.ipynb) — Doob martingales under McDiarmid.
- [Reinforcement Learning](../../Intro_Mach_Learn/Reinforcement_Learning.ipynb) — MDPs: Markov chains you get to steer.